# 1. Imports and config

In [1]:
import re
import pandas as pd
from pathlib import Path

DIR_RAW = Path("../data/raw/")
DIR_OUT = Path("../data/processed")
DIR_OUT.mkdir(parents=True, exist_ok=True)

# Chave = nome exato da subpasta
CIDADES = {
    "A515_Varginha": {"codigo": "A515", "cidade": "Varginha"},
    "A523_Patrocinio": {"codigo": "A523", "cidade": "Patrocinio"},
    "A556_Manhuacu": {"codigo": "A556", "cidade": "Manhuacu"},
    "A567_Machado": {"codigo": "A567", "cidade": "Machado"},
}

ANO_INICIO = 2019  # adicione esta linha na Célula 1

# 2. Read CSV

In [2]:
# Nomes canônicos que queremos ao final (independente do ano do arquivo)
RENAME_COLS = {
    # data/hora — variam entre anos
    "DATA (YYYY-MM-DD)": "data",
    "Data": "data",
    "HORA (UTC)": "hora_utc",
    "Hora UTC": "hora_utc",
    # variáveis meteorológicas (os caracteres corrompidos viram \ufffd no utf-8)
    # usamos posição 2–18 para garantir robustez; veja abaixo
}

# Mapeamento por posição (col index → nome limpo) — imune à corrupção de encoding
COLS_BY_POS = {
    0: "data",
    1: "hora_utc",
    2: "precip_mm",
    3: "pressao_mB",
    4: "pressao_max_mB",
    5: "pressao_min_mB",
    6: "radiacao_kJm2",
    7: "temp_ar_C",
    8: "temp_orvalho_C",
    9: "temp_max_C",
    10: "temp_min_C",
    11: "temp_orvalho_max_C",
    12: "temp_orvalho_min_C",
    13: "umidade_max_pct",
    14: "umidade_min_pct",
    15: "umidade_pct",
    16: "vento_direcao_gr",
    17: "vento_rajada_ms",
    18: "vento_vel_ms",
}


def ler_csv_inmet(filepath: Path, cidade: str, codigo: str) -> pd.DataFrame:
    df = pd.read_csv(
        filepath,
        sep=";",
        encoding="utf-8",
        decimal=",",
        na_values=["-9999", "-9999.0", ""],
        header=0,
        dtype=str,
    )

    # Remove coluna fantasma do ponto-e-vírgula final
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # Renomeia por posição — imune a variações de nome entre anos
    rename_map = {df.columns[i]: nome for i, nome in COLS_BY_POS.items()
                  if i < len(df.columns)}
    df = df.rename(columns=rename_map)

    # Remove linhas sem data válida
    df = df.dropna(subset=["data"])

    # Normaliza formato de data: "2019/01/01" → "2019-01-01"
    df["data"] = df["data"].str.replace("/", "-", regex=False)
    df = df[df["data"].str.match(r"\d{4}-\d{2}-\d{2}", na=False)]

    # Normaliza formato de hora: "0000 UTC" → "0000" | "00:00" → "0000"
    df["hora_utc"] = (
        df["hora_utc"]
        .str.replace(" UTC", "", regex=False)
        .str.replace(":", "", regex=False)
        .str.zfill(4)
    )

    # Constrói datetime UTC → BRT
    df["datetime_utc"] = pd.to_datetime(
        df["data"] + " " + df["hora_utc"],
        format="%Y-%m-%d %H%M",
        errors="coerce",
    )
    df["datetime_brt"] = df["datetime_utc"] - pd.Timedelta(hours=3)
    df = df.dropna(subset=["datetime_utc"])

    # Converte numéricos
    cols_num = list(COLS_BY_POS.values())[2:]
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].str.replace(",", ".", regex=False), errors="coerce"
            )

    df["cidade"] = cidade
    df["codigo_wmo"] = codigo

    cols_final = ["datetime_utc", "datetime_brt", "cidade", "codigo_wmo"] + cols_num
    cols_final = [c for c in cols_final if c in df.columns]

    return df[cols_final].sort_values("datetime_utc").reset_index(drop=True)

# 3. City CSV to daily CSV

In [3]:
def processar_cidade(subpasta: str, meta: dict) -> pd.DataFrame:
    """
    Lê todos os CSVs anuais da subpasta, concatena e agrega para diário.
    """
    cidade = meta["cidade"]
    codigo = meta["codigo"]
    pasta = DIR_RAW / subpasta

    csvs = sorted(
        f for f in pasta.iterdir()
        if f.suffix.upper() == ".CSV"
        and int(re.search(r"(\d{4})\.CSV$", f.name, re.IGNORECASE).group(1)) >= ANO_INICIO)

    if not csvs:
        print(f"  [{cidade}] Nenhum CSV encontrado em {pasta}")
        return pd.DataFrame()

    print(f"\n[{cidade}] {len(csvs)} arquivo(s) encontrado(s):")
    frames = []
    for f in csvs:
        try:
            df = ler_csv_inmet(f, cidade, codigo)
            frames.append(df)
            print(f"    \u2713 {f.name}  \u2192 {len(df):,} linhas horárias")
        except Exception as e:
            print(f"    \u2717 {f.name}  \u2192 ERRO: {e}")

    if not frames:
        return pd.DataFrame()

    df_horario = pd.concat(frames, ignore_index=True)

    # Remove duplicatas (overlap entre arquivos)
    df_horario = (df_horario
                  .drop_duplicates(subset=["datetime_utc"])
                  .sort_values("datetime_utc")
                  .reset_index(drop=True))

    # \u2500\u2500 Agregação diária \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    df_horario["data"] = df_horario["datetime_brt"].dt.normalize()

    agg = {
        "temp_ar_C": ["mean"],
        "temp_max_C": ["max"],
        "temp_min_C": ["min"],
        "temp_orvalho_C": ["mean"],
        "precip_mm": ["sum"],
        "radiacao_kJm2": ["sum"],
        "umidade_pct": ["mean"],
        "umidade_max_pct": ["max"],
        "umidade_min_pct": ["min"],
        "pressao_mB": ["mean"],
        "vento_vel_ms": ["mean", "max"],
        "vento_rajada_ms": ["max"],
    }
    agg = {k: v for k, v in agg.items() if k in df_horario.columns}

    df_diario = (df_horario
                 .groupby(["data", "cidade", "codigo_wmo"])
                 .agg(agg)
                 .reset_index())

    # Achata MultiIndex de colunas
    df_diario.columns = [
        "_".join(c).rstrip("_") if isinstance(c, tuple) else c
        for c in df_diario.columns
    ]

    # \u2500\u2500 Variáveis de risco de geada (CORRIGIDO) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    # JUSTIFICATIVA: a geada agronômica que danifica o cafeeiro ("geada de
    # canopy"/"geada branca") ocorre por irradiação noturna. A folha e o
    # gramado ficam vários graus mais frios que o ar medido a 2 m pela
    # estação automática. O mercado e a literatura tratam ~5 °C a 2 m como
    # limiar de RISCO para Arábica, NÃO 0 °C. Verificação empírica: na geada
    # de jul/2021 (a pior em 25+ anos), a mínima a 2 m em Varginha foi 0,8 °C
    # e a série inteira não tem dias < 0 °C. Um limiar de 0 °C zera o sinal.
    T_RISCO_GEADA = 5.0   # °C a 2 m — limiar de risco agronômico
    if "temp_min_C_min" in df_diario.columns:
        # Flag binária de risco
        df_diario["geada_risco"] = (df_diario["temp_min_C_min"] <= T_RISCO_GEADA).astype(int)
        # Intensidade contínua de frio ("graus-frio"): quanto a mínima ficou
        # abaixo do limiar de risco. 0 em dias quentes; cresce com a severidade.
        df_diario["graus_frio"] = (T_RISCO_GEADA - df_diario["temp_min_C_min"]).clip(lower=0)

    df_diario["data"] = pd.to_datetime(df_diario["data"])

    print(f"    \u2192 {len(df_diario):,} dias  |  "
          f"{df_diario['data'].min().date()} a {df_diario['data'].max().date()}"
          f"  |  dias com risco de geada: {int(df_diario.get('geada_risco', pd.Series()).sum())}")

    return df_diario

# 4. Save individual CSV for each city

In [4]:
dfs_cidades = {}

for subpasta, meta in CIDADES.items():
    df = processar_cidade(subpasta, meta)
    if df.empty:
        continue

    dfs_cidades[meta["cidade"]] = df

    # Salva CSV individual por cidade
    path_out = DIR_OUT / f"inmet_{meta['cidade'].lower()}_diario.csv"
    df.to_csv(path_out, index=False)
    print(f"    Salvo → {path_out}")


[Varginha] 7 arquivo(s) encontrado(s):
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2019_A_31-12-2019.CSV  → 8,760 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2020_A_31-12-2020.CSV  → 8,784 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2021_A_31-12-2021.CSV  → 8,760 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2022_A_31-12-2022.CSV  → 8,760 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2023_A_31-12-2023.CSV  → 8,760 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2024_A_31-12-2024.CSV  → 8,784 linhas horárias
    ✓ INMET_SE_MG_A515_VARGINHA_01-01-2025_A_31-12-2025.CSV  → 8,760 linhas horárias
    → 2,558 dias  |  2018-12-31 a 2025-12-31  |  dias com risco de geada: 13
    Salvo → ../data/processed/inmet_varginha_diario.csv

[Patrocinio] 7 arquivo(s) encontrado(s):
    ✓ INMET_SE_MG_A523_PATROCINIO_01-01-2019_A_31-12-2019.CSV  → 8,760 linhas horárias
    ✓ INMET_SE_MG_A523_PATROCINIO_01-01-2020_A_31-12-2020.CSV  → 8,784 linhas horárias
    ✓ INMET_SE_M

# 5. Concat

In [5]:
df_todas = pd.concat(dfs_cidades.values(), ignore_index=True)
df_todas = df_todas.sort_values(["cidade", "data"]).reset_index(drop=True)

path_all = DIR_OUT / "inmet_todas_cidades_diario.csv"
df_todas.to_csv(path_all, index=False)

print(f"\nCSV consolidado \u2192 {path_all}")
print(f"Shape total: {df_todas.shape}")
print()

# Resumo por cidade
resumo = (df_todas
          .groupby("cidade")
          .agg(
    inicio=("data", "min"),
    fim=("data", "max"),
    dias=("data", "count"),
    dias_risco_geada=("geada_risco", "sum") if "geada_risco" in df_todas.columns else ("data", "count"),
    graus_frio_max=("graus_frio", "max") if "graus_frio" in df_todas.columns else ("data", "count"),
    missing_temp=("temp_ar_C_mean", lambda x: x.isna().mean() * 100),
    missing_precip=("precip_mm_sum", lambda x: x.isna().mean() * 100),
)
          .round(1))

print(resumo.to_string())


CSV consolidado → ../data/processed/inmet_todas_cidades_diario.csv
Shape total: (10232, 18)

               inicio        fim  dias  dias_risco_geada  graus_frio_max  missing_temp  missing_precip
cidade                                                                                                
Machado    2018-12-31 2025-12-31  2558                 3             1.2           2.4             0.0
Manhuacu   2018-12-31 2025-12-31  2558                 1             0.4           1.9             0.0
Patrocinio 2018-12-31 2025-12-31  2558                20             5.2           7.1             0.0
Varginha   2018-12-31 2025-12-31  2558                13             4.2           3.1             0.0


/tmp/ipykernel_167318/1995969398.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(1))
